# 06 — Jaccard Range Overlap (Small-Scale Testing)

Computes Jaccard range overlap for all plant × pollinator species pairs
across the top 50 plant species and all observed pollinator species.
Returns the top 100 pairs by Jaccard score.

## Why Jaccard, not raw co-occurrence counts?

The initial spatial binning analysis (notebook 04) ranked bins by a
combined observation density score:

```
score = (plant_count / max_plant_count) × (pollinator_count / max_pollinator_count)
```

This score is **heavily biased toward high-observation-density areas**.
Both plant and pollinator records in GBIF/PhenoField are primarily sourced
from iNaturalist — an observation platform where contribution density tracks
human population density. The top-ranked bins under this score cluster around
urban areas (Los Angeles, San Francisco, New York corridor, Chicago) where
observers are most active, not where ecologically meaningful plant-pollinator
co-occurrence is highest.

A species with a broad geographic range will accumulate many more total
observations than a range-restricted specialist, making the density score
systematically favor common generalists in accessible locations.

## The Jaccard correction

The Jaccard index measures the fraction of the combined range where both
species are present:

```
Jaccard(plant, pollinator) = |bins_plant ∩ bins_pollinator| / |bins_plant ∪ bins_pollinator|
```

By normalizing by the union, Jaccard controls for range size. A pair where
both species have large ranges but only overlap in a few bins will receive
a low Jaccard score, even if those bins have high observation counts. A
pair where both species are range-restricted but consistently co-occur will
receive a high score.

This was an independent methodological contribution to the pipeline —
developed after the density-score approach revealed systematic bias toward
observer-dense urban areas.

**Output:** `top100_jaccard_pairs.csv`

In [ ]:
import pandas as pd
import numpy as np
import heapq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

BASE       = Path("/scratch/ariana.l")
PLANTS_IN  = BASE / "Plant Pollinator Initial Analysis" / "plant_flowering_events.parquet"
POLL_IN    = BASE / "Plant Pollinator Initial Analysis" / "pollinator_observations_v2.csv"
OUT_DIR    = BASE

BIN_SIZE    = 0.5
MIN_TOTAL   = 10    # minimum total observations to include a species
TOP_N       = 100

print("Paths OK")

In [ ]:
# Load data
plants = pd.read_parquet(PLANTS_IN)
pollinators = pd.read_csv(POLL_IN, low_memory=False)

# Add spatial bins
plants['lat_bin'] = (np.floor(plants['lat'] / BIN_SIZE) * BIN_SIZE).round(1)
plants['lon_bin'] = (np.floor(plants['lon'] / BIN_SIZE) * BIN_SIZE).round(1)
pollinators['lat_bin'] = (np.floor(pollinators['lat'] / BIN_SIZE) * BIN_SIZE).round(1)
pollinators['lon_bin'] = (np.floor(pollinators['lon'] / BIN_SIZE) * BIN_SIZE).round(1)

# Filter species below minimum observation threshold
plant_totals = plants['species'].value_counts()
keep_plants = plant_totals[plant_totals >= MIN_TOTAL].index
plants = plants[plants['species'].isin(keep_plants)]

pollinator_totals = pollinators['pollinator_species'].value_counts()
keep_pollinators = pollinator_totals[pollinator_totals >= MIN_TOTAL].index
pollinators = pollinators[pollinators['pollinator_species'].isin(keep_pollinators)]

print(f"Plants kept:     {plants['species'].nunique()} species")
print(f"Pollinators kept: {pollinators['pollinator_species'].nunique():,} species")

In [ ]:
# Build per-species bin sets
plant_bin_sets = (
    plants.groupby('species')
    .apply(lambda df: set(zip(df['lat_bin'], df['lon_bin'])))
    .to_dict()
)
pollinator_bin_sets = (
    pollinators.groupby('pollinator_species')
    .apply(lambda df: set(zip(df['lat_bin'], df['lon_bin'])))
    .to_dict()
)

print(f"Plant species with bin sets:     {len(plant_bin_sets)}")
print(f"Pollinator species with bin sets: {len(pollinator_bin_sets):,}")

In [ ]:
# Build per-bin observation count dicts for hotspot lookup
plant_counts = (
    plants.groupby(['species', 'lat_bin', 'lon_bin'])
    .size().reset_index(name='plant_count')
)
pollinator_counts = (
    pollinators.groupby(['pollinator_species', 'lat_bin', 'lon_bin'])
    .size().reset_index(name='pollinator_count')
)

plant_count_dict = {
    (row.species, row.lat_bin, row.lon_bin): row.plant_count
    for row in plant_counts.itertuples()
}
pollinator_count_dict = {
    (row.pollinator_species, row.lat_bin, row.lon_bin): row.pollinator_count
    for row in pollinator_counts.itertuples()
}

print("Lookup dicts built.")

In [ ]:
# Compute Jaccard for all plant × pollinator pairs
# Keep top 100 by Jaccard score using a min-heap
#
# Jaccard(A, B) = |bins_A ∩ bins_B| / |bins_A ∪ bins_B|
# Normalizes by union: controls for range size and observation density bias.

print("Computing Jaccard range overlap for all pairs...")
jaccard_heap = []

for i, (p_species, p_bins) in enumerate(plant_bin_sets.items()):
    for q_species, q_bins in pollinator_bin_sets.items():
        intersection = p_bins & q_bins
        if not intersection:
            continue
        jaccard = len(intersection) / len(p_bins | q_bins)

        # Hotspot: shared bin with highest min-count
        hotspot = max(
            intersection,
            key=lambda b: min(
                plant_count_dict.get((p_species, b[0], b[1]), 0),
                pollinator_count_dict.get((q_species, b[0], b[1]), 0)
            )
        )

        entry = (jaccard, len(intersection), p_species, q_species, hotspot[0], hotspot[1])
        if len(jaccard_heap) < TOP_N:
            heapq.heappush(jaccard_heap, entry)
        elif jaccard > jaccard_heap[0][0]:
            heapq.heapreplace(jaccard_heap, entry)

    if (i + 1) % 5 == 0:
        print(f"  plant species {i+1} / {len(plant_bin_sets)} done")

print(f"Done. Top {len(jaccard_heap)} pairs found.")

In [ ]:
# Convert to DataFrame
jaccard_results = pd.DataFrame(
    jaccard_heap,
    columns=['jaccard', 'shared_bins', 'species', 'pollinator_species', 'hotspot_lat', 'hotspot_lon']
).sort_values('jaccard', ascending=False).reset_index(drop=True)

jaccard_results.insert(0, 'rank', jaccard_results.index + 1)
jaccard_results['jaccard'] = jaccard_results['jaccard'].round(4)

out_path = OUT_DIR / 'top100_jaccard_pairs.csv'
jaccard_results.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
jaccard_results.head(20)

In [ ]:
# Visualization: hotspot locations
fig, ax = plt.subplots(figsize=(13, 8))
scatter = ax.scatter(
    jaccard_results['hotspot_lon'],
    jaccard_results['hotspot_lat'],
    c=jaccard_results['jaccard'],
    s=jaccard_results['shared_bins'] / 5,
    cmap='viridis', alpha=0.8,
    edgecolors='black', linewidths=0.4
)
plt.colorbar(scatter, ax=ax, label='Jaccard score')
ax.set_xlim(-126, -66)
ax.set_ylim(24, 50)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(
    'Top 100 Plant-Pollinator Pairs: Hotspot Locations\n'
    '(color = Jaccard score, size = shared bin count)\n'
    'Jaccard controls for range size and observation density bias'
)
plt.tight_layout()
plt.savefig(OUT_DIR / 'top100_jaccard_hotspot_map.png', dpi=150)
print("Saved.")